# Tweety-3 — Description Logics en C#/.NET (port natif IKVM)

> **Serie Tweety — port C#/.NET natif (EPIC [#4667](https://github.com/jsboige/CoursIA/issues/4667)).**
> Ce notebook exploite le module **`logics-dl`** de [TweetyProject](https://tweetyproject.org/) **sans JVM** :
> la librairie Java est compilee vers un fat-jar Maven shade puis executee sur le runtime .NET via
> [IKVM](https://github.com/ikvm-refined/ikvm).

Navigation : [Tweety-2-Basic-Logics-Csharp](Tweety-2-Basic-Logics-Csharp.ipynb) (propositionnel) ·
[Tweety-2b-Semantics-Csharp](Tweety-2b-Semantics-Csharp.ipynb) (mondes possibles) ·
[Tweety-2c-FOL-Csharp](Tweety-2c-FOL-Csharp.ipynb) (premier ordre) ·
[Tweety-3-Dung-Csharp](Tweety-3-Dung-Csharp.ipynb) (argumentation abstraite) ·
**Tweety-3-Advanced-Logics (ce notebook — DL)**.

---

## Objectifs pedagogiques

Les **Description Logics** (DL) sont le fragment decidable de la logique du premier ordre qui sous-tend les **ontologies** (web semantique, OWL, bases de connaissances medicales ou industrielles). Elles equilibrent expressivite et decidabilite via une syntaxe compacte dediee aux **concepts**, **roles** et **individus**.

Dans ce notebook on manipule :

- la **TBox** : ensemble d'axiomes terminologiques (definitions de concepts, inclusions) ;
- la **ABox** : ensemble d'assertions sur les individus (appartenance a un concept, relations de role) ;
- le **reasoner `NaiveDlReasoner`** : un raisonneur naif (mais fonctionnel) qui repond aux requetes de subsomption et d'instance ;
- le **parser `DlParser`** : analyse syntaxique des expressions DL classiques (AL, ALC, SHIQ simplifie).

L'objectif : faire valoir la specificite des DL face au propositionnel (Tweety-2) et a l'abstrait (Dung) — raisonner sur des **hierarchies de concepts** plutot que sur des booleens ou des graphes d'attaque.


## 1 — Runtime IKVM : charger le module `logics-dl`

On installe le runtime IKVM, on fusionne l'image (base + arch), puis on charge la DLL
`org.tweetyproject.tweety-advanced-logics.dll` (compilee cote build a partir d'un fat-jar shade embarquant
`logics-dl` + ses dependances transitives : `logics-commons`, `fol`, `pl`, `math`, `commons`, `sat4j`, `commons-math`).


In [ ]:
#r "nuget: IKVM, 8.15.0"
#r "nuget: IKVM.Image, 8.15.0"
#r "nuget: IKVM.Image.runtime.win-x64, 8.15.0"


In [ ]:
using System.IO;
string ikvmVer = "8.15.0", ikvmRid = "win-x64";
string nugetRoot = Environment.GetEnvironmentVariable("NUGET_PACKAGES")
    ?? Path.Combine(Environment.GetFolderPath(Environment.SpecialFolder.UserProfile), ".nuget", "packages");
string ikvmBaseAny = Path.Combine(nugetRoot, "ikvm.image", ikvmVer, "ikvm", "any", "any");
string ikvmArchDir = Path.Combine(nugetRoot, "ikvm.image.runtime." + ikvmRid, ikvmVer, "ikvm", "any", ikvmRid);
string ikvmHome = Path.Combine(Path.GetTempPath(), "ikvm-home-" + ikvmVer + "-" + ikvmRid);
void IkvmCopyMerge(string src, string dst)
{
    foreach (var d in Directory.GetDirectories(src, "*", SearchOption.AllDirectories))
        Directory.CreateDirectory(d.Replace(src, dst));
    foreach (var f in Directory.GetFiles(src, "*", SearchOption.AllDirectories))
    {
        var t = f.Replace(src, dst);
        Directory.CreateDirectory(Path.GetDirectoryName(t));
        File.Copy(f, t, overwrite: true);
    }
}
if (Directory.Exists(ikvmBaseAny) && Directory.Exists(ikvmArchDir))
{
    Directory.CreateDirectory(ikvmHome);
    IkvmCopyMerge(ikvmBaseAny, ikvmHome);
    IkvmCopyMerge(ikvmArchDir, ikvmHome);
}
AppContext.SetData("IKVM.Home", ikvmHome);
Console.WriteLine("IKVM home=" + (File.Exists(Path.Combine(ikvmHome, "lib", "tzdb.dat")) ? "OK" : "MISSING"));


In [ ]:
#r "org.tweetyproject.tweety-advanced-logics.dll"


In [ ]:
// Verification que la DLL chargee expose bien les classes DL cles.
using System.Reflection;
var tweetyDll = "org.tweetyproject.tweety-advanced-logics.dll";
var an = AssemblyName.GetAssemblyName(tweetyDll);
Console.WriteLine($"Tweety (IKVM) reference chargee : {an.Name} v{an.Version} ({new FileInfo(tweetyDll).Length / 1024 / 1024:F1} Mo).");


## 2 — Bases de connaissances DL : TBox et ABox

Une **base de connaissances DL** se decompose en deux parties :

- la **TBox** (*terminological box*) declare les concepts et leurs relations :
  inclusions `A ⊑ B` ("tout A est un B"), equivalences `A ≡ B` ;
- la **ABox** (*assertional box*) declare les individus :
  appartenance `a : C` ("l'individu `a` est une instance du concept `C`") ou relations `a R b` ("`a` est en relation `R` avec `b`").

En Tweety, ces deux pieces sont exposees via :

- `DlBeliefSet` : contient a la fois TBox et ABox (structure unifiee) ;
- `DlParser` : analyse de strings comme `"Human ⊑ Mortal"`, `"alice : Human"` ;
- `Concept`, `Role`, `Individual` : noeuds syntaxiques des axiomes.

Pour ce notebook on travaille sur la logique **ALC** (Attributes Language with Complement) qui admet :
 - les concepts atomiques `A` ;
 - les booleens `⊤` (top), `⊥` (bottom), `¬C` (negation), `C ⊓ D` (intersection), `C ⊔ D` (union) ;
 - les restrictions `forall R.C` et `exists R.C`.


In [ ]:
// Construire un concept compose via les static factories :
//   Concept.TOP / Concept.BOTTOM / Concept.NOT(C) / Concept.AND(C1, C2) /
//   Concept.OR(C1, C2) / Concept.ALL(R, C) / Concept.EXISTS(R, C).
using org.tweetyproject.logics.dl.syntax;
var human = new Concept("Human");
var mortal = new Concept("Mortal");
var humanMortal = Concept.AND(human, mortal);
Console.WriteLine("Concept compose = " + humanMortal);
Console.WriteLine("Variables humaines : "); for (var v in human.getVariables()) Console.WriteLine("  " + v);


In [ ]:
// Parser : construire TBox + ABox a partir de strings.
// Syntaxe :
//   - "Human ⊑ Mortal"        => TBox : tout Human est Mortal (inclusion)
//   - "alice : Human"             => ABox : alice est un Human
//   - "alice friend bob"          => ABox : alice en role friend avec bob
using org.tweetyproject.logics.dl.parser;
using org.tweetyproject.logics.dl.syntax;
var parser = new DlParser();
var kb = new DlBeliefSet();
kb.add(parser.parseFormula("Human ⊑ Mortal"));   // TBox : axiom d'inclusion
kb.add(parser.parseFormula("alice : Human"));       // ABox : alice = Human
kb.add(parser.parseFormula("bob : Human"));         // ABox : bob = Human
Console.WriteLine("KB DL = " + kb);
Console.WriteLine("nb formules KB = " + kb.size());


## 3 — Raisonner avec `NaiveDlReasoner`

`NaiveDlReasoner` est un raisonneur naif de subsomption et d'instance :

- `query(DlBeliefSet, Formula)` prend une formule DL (concept / axiome) et repond si la KB l'implique (model-theoretic entailment).
- Sa complexite est exponentielle (il enumere les interpretations) ; il sert surtout a valider la mechanique, pas pour la production.

**Cas pedagogique canonique** :
1. KB = `{ Human ⊑ Mortal, alice : Human }` -> `alice : Mortal` doit etre entail (chainage direct).
2. KB = `{ ... ABox mentionne alice et bob, ... }` -> verifier la consistance (pas d'incoherence sensee).


In [ ]:
// Test d'inference : alice : Human et Human ⊑ Mortal doivent impliquer alice : Mortal.
using org.tweetyproject.logics.dl.parser;
using org.tweetyproject.logics.dl.syntax;
using org.tweetyproject.logics.dl.reasoner;
var parser = new DlParser();
var kb = new DlBeliefSet();
kb.add(parser.parseFormula("Human ⊑ Mortal"));
kb.add(parser.parseFormula("alice : Human"));
var reasoner = new NaiveDlReasoner();
var query = parser.parseFormula("alice : Mortal");
bool entail = reasoner.query(kb, query);
Console.WriteLine($"KB entail (alice : Mortal) : {entail}");

// Variante negative : Bob n'est pas Human donc on ne peut pas deduire Bob : Mortal.
var kb2 = new DlBeliefSet();
kb2.add(parser.parseFormula("Human ⊑ Mortal"));
kb2.add(parser.parseFormula("bob : Human"));
var queryBobMort = parser.parseFormula("bob : Human");   // sanity : vrai
var queryBotZombie = parser.parseFormula("bob : Zombie"); // sanity : faux (Zombie pas declare)
Console.WriteLine($"KB2 entail (bob : Human) : {reasoner.query(kb2, queryBobMort)}");
Console.WriteLine($"KB2 entail (bob : Zombie) : {reasoner.query(kb2, queryBotZombie)}");


---

## Exercices

> Stubs **sans `throw`/`raise`** (convention C.1) : le notebook s'execute de bout en bout meme non complete.


### Exercice 1 — Chaine de subsomption a 3 niveaux

Construisez une TBox a 3 concepts en chaine : `Salarie ⊑ Personne ; Employe ⊑ Personne`.
Verifiez avec `NaiveDlReasoner.query` que `Salarie ⊑ Personne` est entail, mais que `Salarie ⊑ Employe` ne l'est **pas** (ils sont disjoints en termes de subsomption car aucun n'inclut l'autre dans l'autre sens).

**Indice** : `parser.parseFormula("Salarie ⊑ Personne")` et `parser.parseFormula("Salarie ⊑ Employe")` produisent les axiomes a passer au reasoner. Ces chaines se comparent via `⊑` (subseteq logiques), pas via `≡` (equivalence).


In [ ]:
// TODO etudiant : TBox 3 concepts Salarie/Personne/Employe, queryer la subsomption
object entSalariePersonne = null;   // TODO etudiant : bool, KB entail (Salarie ⊑ Personne)
object entSalarieEmploye = null;    // TODO etudiant : bool, KB entail (Salarie ⊑ Employe)
Console.WriteLine($"KB entail (Salarie ⊑ Personne) : {entSalariePersonne ?? "Exercice a completer"}");
Console.WriteLine($"KB entail (Salarie ⊑ Employe)  : {entSalarieEmploye ?? "Exercice a completer"}");


### Exercice 2 — Role et restriction existentielle

Les **roles** sont les proprietes binaires en DL. Definissez un role `hasParent`, puis testez si la KB
`{ ∃hasParent.Human ⊑ Parent, alice : Parent }`
implique qu'il **existe** un humain lie a alice par `hasParent`.

Cela suppose de manipuler les axiomes de subsomption, ce que le `NaiveDlReasoner` gere via `parseFormula`.

**Question** : la subsomption `∃hasParent.Human ⊑ Parent` dit-elle qu'un Parent est quelqu'un qui a au moins un parent humain. Si alice est un Parent, doit-on conclure qu'il existe un parent humain de alice ?


In [ ]:
// TODO etudiant : KB avec restriction existentielle sur role hasParent.
// Definissez la TBox, ajoutez alice : Parent, et queryez : la KB entail-elle "alice : ∃ hasParent . Human" ?
object existeParentHumain = null;  // TODO etudiant : bool (ou null si non complete)
Console.WriteLine($"KB entail (alice : ∃ hasParent . Human) : {existeParentHumain ?? "Exercice a completer"}");


### Exercice 3 — Cohabitation exemple + exercice (multi-formules)

Demontrer la cohabitation (cf [exercise-example-labeling.md](.claude/rules/exercise-example-labeling.md)) :
ci-dessus on a deja **un exemple resolu** (sections 2-3 : chainage direct alice-Mortal). Ajoutez ici un **exercice non resolu** qui exige l'etudiant de raisonner sur la **conjonction de concepts** :

Construisez un concept `EtudiantSalarie = Human ⊓ Salarie`, et testez si une ABox avec `alice : EtudiantSalarie` est consistante (ne cree pas de contradiction) avec une TBox contenant `Human ⊑ Personne`.

**Indice** : `Concept.AND(humanConcept, salarieConcept)` permet de construire le concept compose, mais ici on peut utiliser directement le parser : `"alice : Human ⊓ Salarie"` n'est pas une formule DL valide en ALC standard ; utilisez `parser.parseBeliefSet` ou composez deux formules distinctes.


In [ ]:
// TODO etudiant : conjonction de concepts (alice : Human ET alice : Salarie),
// verifier la consistance avec une TBox simple.
object estConsistant = null;     // TODO etudiant : bool, KB entail (alice : Human), KB entail (alice : Salarie)
Console.WriteLine($"KB entail (alice : Human)   : {estConsistant ?? "Exercice a completer"}");
Console.WriteLine($"KB entail (alice : Salarie) : {estConsistant ?? "Exercice a completer"}");


---

## Conclusion

On a porte en C#/.NET natif (sans JVM) le module **`logics-dl`** de TweetyProject — le sous-ensemble
Description Logics (ALC) des logiques avancees — via IKVM, complement des ports precedents :

- **Tweety-2-Basic-Logics** : propositionnel (PL)
- **Tweety-2b-Semantics** : mondes possibles
- **Tweety-2c-FOL** : premier ordre
- **Tweety-3-Dung** : argumentation abstraite
- **Tweety-3-Advanced-Logics (ce notebook)** : **Description Logics** (ALC, TBox/ABox, NaiveDlReasoner)

Les DL sont le **pont entre logique et representation de connaissances** : la syntaxe compacte (concepts, roles, axiomes) nourrit les ontologies OWL / web semantique, et fournit un terrain d'exercice pour les raisonneurs specialises.

### Pourquoi un raisonneur `NaiveDlReasoner` ?

Le `NaiveDlReasoner` enumere completement les interpretations possibles du domaine logique.
Sa complexite est exponentielle, donc reservee aux KB pedagogiques de petite taille ; pour la
production on utilise des raisonneurs dedies (Pellet, HermiT, ELK) ou des variantes optimisees
de Tweety (`ShiqReasoner`, `ElReasoner`).

### Limites connues de l'IKVM 8.15.0 avec Java 17

Le fat-jar shaded embarque du bytecode Java compile en version 59 (Java 15+). IKVM 8.15.0 ne
compile pas integralement ces classes (365 `IKVM0101` warnings sur l'operation `dotnet build`).
Les classes DL de surface (`DlBeliefSet`, `DlParser`, `NaiveDlReasoner`, `Concept`) sont
compilees avec succes et directement utilisables. Pour les modules internes (sat4j, FOL,
commons-lang3), les classes non compilees apparaissent comme `IKVM0100` (not found) ; ceci
n'affecte pas le port pedagogique des DL.

### References

- Baader, F., Calvanese, D., McGuinness, D., Nardi, D., Patel-Schneider, P. (eds.). *The Description
  Logic Handbook*. Cambridge University Press, 2003.
- TweetyProject — [logics-dl module](https://tweetyproject.org/api/dl/).
- Port C#/.NET via IKVM — EPIC [#4667](https://github.com/jsboige/CoursIA/issues/4667).
